# Gold Analytics Layer

This notebook transforms the Silver layer into business-ready analytical datasets.

Responsibilities:

- Read Silver Delta Table
- Generate business KPIs
- Create Gold Summary Tables
- Save Gold Delta Tables
- Validate Gold Tables

Output:
- service_performance
- service_health
- log_level_summary
- http_status_summary
- daily_summary
- topic_summary
- anomaly_summary
- service_risk_dashboard

In [0]:
%run ./00_Project_Setup

In [0]:
## Read Silver Table
silver_df = spark.table("`log-analytics`.silver.silver_logs")

print("="*60)
print("SILVER DATASET")
print("="*60)

print("Rows    :", silver_df.count())
print("Columns :", len(silver_df.columns))

display(silver_df.limit(10))

In [0]:
# Validation
display(
    silver_df.groupBy("is_anomaly").count()
)

In [0]:
display(
    silver_df.groupBy("log_level").count()
)

In [0]:
display(
    silver_df.groupBy("kafka_topic").count()
)

In [0]:
# Service Performance Summary
from pyspark.sql import functions as F

service_performance = (
    silver_df
    .groupBy("service")
    .agg(
        F.count("*").alias("total_requests"),
        F.avg("response_time").alias("avg_response_time"),
        F.max("response_time").alias("max_response_time"),
        F.min("response_time").alias("min_response_time"),

        F.avg("status_code").alias("avg_status_code"),

        F.sum(
            F.when(F.col("log_level")=="ERROR",1).otherwise(0)
        ).alias("error_count"),
        F.sum(
            F.when(F.col("log_level")=="WARNING",1).otherwise(0)
        ).alias("warning_count"),
        F.sum(
            F.when(F.col("log_level")=="CRITICAL",1).otherwise(0)
        ).alias("critical_count"),

        F.sum(
            F.when(F.col("status_code")==200,1).otherwise(0)
        ).alias("success_count"),

        F.sum("is_anomaly").alias("anomaly_count")
    )
    .withColumn(
        "error_rate",
        F.round(
            F.col("error_count")/F.col("total_requests")*100, 2)
    )

    .withColumn(
        "warning_rate",
        F.round(F.col("warning_count")/F.col("total_requests")*100,2)
    )

    .withColumn(
        "anomaly_rate",
        F.round(F.col("anomaly_count")/F.col("total_requests")*100,2)
    )
)

display(service_performance.orderBy(F.desc("error_count")))

In [0]:
# Calculate Error Rate
service_performance = (
    service_performance
    .withColumn(
        "error_rate",
        F.round(
            (F.col("error_count")/F.col("total_requests"))*100,
            2
        )
    )
)

In [0]:
# Display
display(
    service_performance.orderBy(
        F.desc("error_count")
    )
)

In [0]:
# service health
service_health = (
    service_performance
    .withColumn(
        "health_status",
        F.when(
            F.col("error_rate")<1,
            "Healthy"
        )
        .when(
            (F.col("error_rate")>=1) & (F.col("error_rate")<5),
            "Degraded"
        )
        .otherwise("Critical")
    )
)

display(service_health)

In [0]:
## Log Level Summary
total_logs = silver_df.count()

log_level_summary = (
    silver_df
    .groupBy("log_level")
    .count()
    .withColumn(
        "percentage",
        F.round(F.col("count")/F.lit(total_logs)*100, 2)
    )
)

display(log_level_summary)

In [0]:
# HTTP Status Summary
http_status_summary = (
    silver_df
    .groupBy("status_code")
    .agg(
        F.count("*").alias("total_requests"),
        F.avg("response_time").alias("avg_response_time")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("total_requests")/F.lit(total_logs)*100,
            2
        )
    )
)

display(http_status_summary)

In [0]:
## Daily Summary
daily_summary = (
    silver_df
    .groupBy("event_date")
    .agg(
        F.count("*").alias("total_logs"),
        F.sum(
            F.when(F.col("log_level") == "ERROR",1).otherwise(0)
        ).alias("errors"),
         F.sum(
            F.when(F.col("log_level") == "WARNING",1).otherwise(0)
        ).alias("warnings"),
         F.sum(
            F.when(F.col("log_level") == "CRITICAL",1).otherwise(0)
        ).alias("critical"),
        F.sum(
            F.when(F.col("is_anomaly") == 1,1).otherwise(0)
        ).alias("anomalies"),
        F.avg("response_time").alias("avg_response_time")
    )
)
display(daily_summary)

In [0]:
## Kafka Topic Summary
topic_summary = (
    silver_df
    .groupBy("kafka_topic")
    .agg(
        F.count("*").alias("total_logs"),
        F.avg("response_time").alias("avg_response_time"),
        F.sum(
            F.when(F.col("log_level")=="ERROR",1).otherwise(0)
        ).alias("error_count"),
        F.sum(
            F.when(F.col("log_level")=="WARNING",1).otherwise(0)
        ).alias("warning_count"),
        F.sum(
            F.when(F.col("is_anomaly")==1,1).otherwise(0)
        ).alias("anomaly_count")
    )
)
display(topic_summary)

In [0]:
# Anomaly Summary
anomaly_summary = (
    silver_df
    .groupBy("service")
    .agg(
        F.sum(
            F.when(F.col("is_anomaly")==1,1).otherwise(0)
        ).alias("anomaly_count"),
        F.avg("response_time").alias("avg_response_time"),
        F.sum(
            F.when(F.col("log_level") == "ERROR",1).otherwise(0)
        ).alias("error_count")
    )
)
display(anomaly_summary)

In [0]:
# Service Risk Dashboard
service_risk_dashboard = (
    service_health
    .withColumn(
        "risk_score",
        F.round(
            (
                F.col("error_rate")*0.5 +
                F.col("anomaly_rate")*0.3 +
                F.col("avg_response_time")*20*0.2
            ),
            2
        )
    )
)

display(service_risk_dashboard.orderBy(F.desc("risk_score")))

In [0]:
## Save Gold Tables
service_performance.write.mode("overwrite").saveAsTable(
    "`log-analytics`.gold.service_performance"
)

service_health.write.mode("overwrite").saveAsTable(
    "`log-analytics`.gold.service_health"
)

log_level_summary.write.mode("overwrite").saveAsTable(
    "`log-analytics`.gold.log_level_summary"
)

http_status_summary.write.mode("overwrite").saveAsTable(
    "`log-analytics`.gold.http_status_summary"
)

daily_summary.write.mode("overwrite").saveAsTable(
    "`log-analytics`.gold.daily_summary"
)

topic_summary.write.mode("overwrite").saveAsTable(
    "`log-analytics`.gold.topic_summary"
)

anomaly_summary.write.mode("overwrite").saveAsTable(
    "`log-analytics`.gold.anomaly_summary"
)

service_risk_dashboard.write.mode("overwrite").saveAsTable(
    "`log-analytics`.gold.service_risk_dashboard"
)

In [0]:
## Validation
gold_tables = [
    "service_performance",
    "service_health",
    "log_level_summary",
    "http_status_summary",
    "daily_summary",
    "topic_summary",
    "anomaly_summary",
    "service_risk_dashboard"
]

for table in gold_tables:
    df = spark.table(f"`log-analytics`.gold.{table}")

    print("="*70)
    print(table.upper())
    print("="*70)

    print("Rows :", df.count())

    print("Columns :", len(df.columns))

    display(df.limit(5))

In [0]:
%sql
-- SQL Validation

SELECT 
    service,
    total_requests,
    error_rate,
    anomaly_rate
FROM `log-analytics`.gold.service_performance
ORDER BY error_rate DESC;


In [0]:
%sql
SELECT *
FROM `log-analytics`.gold.service_health;

In [0]:
%sql
SELECT *
FROM `log-analytics`.gold.topic_summary;

In [0]:
print("="*70)
print("GOLD LAYER COMPLETED")
print("="*70)

print("Source : Silver Layer")

print("Business Tables Created : 8")

print("Gold Layer Status : SUCCESS")